In [ ]:
#importa??o de bibliotecas
import tensorflow as tf
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# 1. Carregar o dataset balanceado

In [ ]:
df = pd.read_csv('dataset_pronto_treino.csv')

# 2. Separar Features (Deltas) e Alvo (y)
X = df[['delta_t', 'delta_u']].values
y = df['classe'].values

# 3. Divis?o de Treino e Teste (80% / 20%)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.preprocessing import StandardScaler
scaler_treino = StandardScaler()
X_train_scaled = scaler_treino.fit_transform(X_train)
X_test_scaled = scaler_treino.transform(X_test)

# 4. Arquitetura da Rede Neural (TinyML)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(2,)), # Entrada: Delta T e Delta U
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 5. Treinamento

In [ ]:
print("Iniciando o treinamento...")
history = model.fit(X_train_scaled, y_train, epochs=100, batch_size=16,
                    validation_split=0.1, verbose=1)

# 6. Avalia??o

In [ ]:
loss, acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\n? Acur?cia final nos dados de teste: {acc*100:.2f}%")

# 7. Convers?o para TensorFlow Lite (Formato para microcontrolador)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Salvar o arquivo bin?rio
with open('modelo_predator.tflite', 'wb') as f:
    f.write(tflite_model)

print("\n?? Arquivo 'modelo_predator.tflite' gerado com sucesso!")

#8.Gera??o do Arquivo de Cabe?alho C

In [ ]:
import os

def convert_to_c_array(tflite_path, model_name="modelo_predator"):
    with open(tflite_path, 'rb') as f:
        tflite_content = f.read()

    hex_lines = [f'0x{byte:02x}' for byte in tflite_content]

    c_str = f'#ifndef {model_name.upper()}_H\n'
    c_str += f'#define {model_name.upper()}_H\n\n'
    c_str += f'// Tamanho do modelo: {len(tflite_content)} bytes\n'
    c_str += f'const unsigned int {model_name}_len = {len(tflite_content)};\n\n'
    c_str += f'const unsigned char {model_name}[] __attribute__((aligned(16))) = {{\n'

    for i in range(0, len(hex_lines), 12):
        c_str += '  ' + ', '.join(hex_lines[i:i+12]) + ',\n'

    c_str += '};\n\n#endif\n'

    with open(f"{model_name}.h", 'w') as f:
        f.write(c_str)
    return f"{model_name}.h"

header_file = convert_to_c_array('modelo_predator.tflite')
print(f"?? Arquivo '{header_file}' pronto para download na pasta lateral do Colab!")